# Zadanie 3: optymalizacja dyskretna

Termin realizacji: 15 kwietnia 2024

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

## Na 3.0

Do realizacji:

1. Zaimplementuj dyskretny problem plecakowy z trzema plecakami w MiniZinc na podstawie przykładu z poprzednich zajęć (plik `minizinc.ipynb`). Spróbuj rozwiązać problem dla 10 zestawów parametrów o różnych wielkościach tak, aby rozwiązanie największego problemu trwało powyżej 5 sekund. Zanotuj w każdym przypadku liczbę wszystkich przedmiotów, pojemności plecaków, liczbę wybranych przedmiotów i sumaryczną wartość przedmiotów w każdym plecaku osobno.
2. Zmodyfikuj metodę z notatnika `tabu_search.ipynb` tak aby rozwiązywała opisywany problem plecakowy. Porównaj na tych samych problemach czy Minizinc i Tabu search zwracają równie dobre rozwiazania, oraz wypisz jakie to są rozwiązania. Wykonaj eksperymenty z trzema różnymi długościami listy zakazów (1, 2, 5).

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Rozszerz możliwe ruchy w tabu search o przeniesienie przedmiotu z jednego plecaka do drugiego. Zapisz rozważ czy to poprawia działanie metody (czy znalezione jest lepsze, takie samo czy gorsze rozwiązanie? czy rozwiązanie jest znajdowane szybciej czy wolniej?). Dla każdego z 10 zestawów parametrów problemu plecakowego wykonaj ocenę przez uśrednienie dla 10 różnych losowych przypadków. Podsumuj dane w formie tabelki z czterema kolumnami (Minizinc, tabu search z listą o długości 1, 2, i 5) oraz 10 wierszami (po jednym dla zestawu parametrów problemu), a w komórkach umieść średnią wartość wartości przedmiotów oraz średni czas potrzebny do uzyskania rozwiązania.

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Zaimplementuj samodzielnie algorytm symulowanego wyżarzania analogicznie do tabu search. Porównaj jego działanie do rozważanych wcześniej rozwiązań dla trzech różnych schematów chłodzenia.


# 1.1 MiniZinc

In [1]:
using Distributions

function make_dzn(n::Int, capacities::Vector{Int}, data_number::Int)
    
    profits = rand(DiscreteUniform(10, 1000), n)
    weights = rand(DiscreteUniform(10, 100), n)
    content = """
    ITEM = _(1..$n);
    capacities = $capacities;
    profits = $profits;
    weights = $weights;
    """
    file = open("generated_data/knapsack_generated_$data_number.dzn", "w+")
    write(file, content)
    close(file)
end

make_dzn (generic function with 1 method)

In [ ]:
n_items = [80, 5, 10, 20, 25, 30, 40, 50, 75, 100]

for (i, n) in enumerate(n_items)
    capacities = rand(DiscreteUniform(100, 1000), 3)
    make_dzn(n, capacities, i)

end

In [29]:
n_items = [80, 5, 10, 20, 25, 30, 40, 50, 75, 100]


for (i, n) in enumerate(n_items)
    println("------------KNAPSACK PROBLEM $(i-1)------------")

    f = open("generated_data/knapsack_generated_$(i-1).dzn", "r")
    for line in readlines(f)
        println(line)
    end
    close(f)

    g = open("knapsack_results/knapsack_$(i-1).txt", "r")
    for line in readlines(g)
        println(line)
    end
    close(g)
    println("--------------------------------------------")

end


------------KNAPSACK PROBLEM 0------------
ITEM = _(1..80);
capacities = [619, 360, 984];
profits = [830, 823, 288, 56, 726, 439, 534, 194, 387, 749, 127, 846, 165, 868, 143, 419, 750, 283, 505, 435, 828, 946, 315, 559, 462, 228, 342, 400, 371, 576, 356, 807, 346, 591, 991, 311, 878, 104, 318, 347, 985, 345, 354, 939, 333, 46, 873, 640, 947, 341, 746, 806, 609, 833, 731, 922, 404, 949, 916, 138, 715, 373, 997, 57, 659, 887, 797, 854, 537, 342, 412, 671, 406, 275, 846, 570, 142, 587, 734, 266];
weights = [61, 75, 95, 19, 77, 20, 75, 12, 49, 90, 24, 48, 84, 100, 33, 19, 20, 41, 100, 38, 64, 91, 16, 35, 45, 47, 45, 83, 98, 41, 13, 71, 46, 68, 59, 62, 19, 79, 48, 59, 100, 17, 31, 45, 40, 75, 57, 83, 57, 79, 35, 69, 40, 94, 39, 65, 11, 56, 63, 21, 22, 22, 59, 79, 13, 10, 72, 59, 52, 59, 52, 27, 92, 55, 44, 42, 64, 32, 85, 68];
Items count = 80
knapsack_1 = {to_enum(ITEM,5), to_enum(ITEM,6), to_enum(ITEM,8), to_enum(ITEM,12), to_enum(ITEM,16), to_enum(ITEM,17), to_enum(ITEM,23), to_enum(ITEM

# 1.2 Tabu Search

In [11]:
using DataStructures
using Distributions

mutable struct TabuState{TMove,P,TF}
    tabu_buffer::CircularBuffer{TMove}
    best_seen::P
    best_seen_obj::TF
    current::P
    considered::P
    iter::Int
end

function TabuState(p, x0; buffer_length::Int=10)
    moves = possible_moves(p, x0)
    obj = objective(p, x0)
    return TabuState{eltype(moves),typeof(x0),typeof(obj)}(
        CircularBuffer{eltype(moves)}(buffer_length), x0, obj, copy(x0), copy(x0), 1
    )
end


function solve_tabu(p, s::TabuState; iteration_limit::Int=100)
    while s.iter < iteration_limit
        moves = possible_moves(p, s.current)
        best_move = 0
        best_move_obj = Inf
        for (i_move, move) in enumerate(moves)
            if in(move, s.tabu_buffer)
                # move forbidden, do not consider
                continue
            end
            # evaluate move
            copyto!(s.considered, s.current)
            apply!(s.considered, move)
            considered_value = objective(p, s.considered)
            if considered_value < best_move_obj
                best_move = i_move
                best_move_obj = considered_value
            end
        end
        # no allowed move found
        if best_move == 0
            break
        end
        apply!(s.current, moves[best_move])
        push!(s.tabu_buffer, invert_move(p, moves[best_move]))
        if best_move_obj < s.best_seen_obj
            # best so far, let's remember it
            copyto!(s.best_seen, s.current)
            s.best_seen_obj = best_move_obj
        end
        s.iter += 1
    end
    return s.best_seen
end

struct MultiKnapsackProblem
    capacities::NTuple{3,Int}
    weights::Vector{Int}
    profits::Vector{Int}
end

function objective(p::MultiKnapsackProblem, x::Vector{Int})
    total_weight = (0, 0, 0)
    total_profit = 0
    for (i, assign) in enumerate(x)
        if assign != 0
            total_weight = ntuple(j -> total_weight[j] + (j == assign ? p.weights[i] : 0), 3)
            total_profit += p.profits[i]
        end
    end
    for i in 1:3
        if total_weight[i] > p.capacities[i]
            return Inf  
        end
    end
    return -total_profit 
end

function apply!(x::Vector{Int}, move::Tuple{Symbol,Int,Int})
    action, item, knapsack = move
    if action == :assign
        x[item] = knapsack
    elseif action == :unassign
        x[item] = 0
    end
    return x
end

function invert_move(p::MultiKnapsackProblem, move::Tuple{Symbol,Int,Int})
    action, item, knapsack = move
    if action == :assign
        return (:unassign, item, knapsack)
    elseif action == :unassign
        return (:assign, item, knapsack)
    end
end

function possible_moves(p::MultiKnapsackProblem, x::Vector{Int})
    move_list = Tuple{Symbol,Int,Int}[]
    current_weights = (0, 0, 0)
    for (i, assign) in enumerate(x)
        if assign != 0
            current_weights = ntuple(j -> current_weights[j] + (j == assign ? p.weights[i] : 0), 3)
        end
    end

    for (i, assign) in enumerate(x)
        if assign == 0
            for k in 1:3
                if current_weights[k] + p.weights[i] <= p.capacities[k]
                    push!(move_list, (:assign, i, k))
                end
            end
        else
            push!(move_list, (:unassign, i, assign))
        end
    end
    return move_list
end

function generate_problem()
    n_items = 100
    profits = rand(DiscreteUniform(10, 1000), n_items)
    weights = rand(DiscreteUniform(10, 100), n_items)
    capacities = (1500, 1500, 1500)
    return MultiKnapsackProblem(capacities, weights, profits)
end

function test(p)
    x0 = fill(0, length(p.weights))  
    st = TabuState(p, x0; buffer_length=20)
    sol = solve_tabu(p, st; iteration_limit=100000)
    knapsack1 = []
    knapsack2 = []
    knapsack3 = []
    not_assigned = []
    for (i, k) in enumerate(sol)
        if k == 1
            push!(knapsack1, i)
        elseif k == 2
            push!(knapsack2, i)
        elseif k == 3
            push!(knapsack3, i)
        else
            push!(not_assigned, i)
        end
    end
    println("Items in knapsack 1: ", knapsack1)
    println("Items in knapsack 2: ", knapsack2)
    println("Items in knapsack 3: ", knapsack3)
    println("Items not assigned: ", not_assigned)
    println("Best objective: ", st.best_seen_obj)
    println("Last iteration: ", st.iter)
end



test (generic function with 1 method)

In [12]:
n_items = [80, 5, 10, 20, 25, 30, 40, 50, 75, 100]

function generate_problems()
    problems = []
    
    for (i, n) in enumerate(n_items)
        capacities = []
        profits = []
        weights = []
        f = open("generated_data/knapsack_generated_$(i-1).dzn", "r")
        for line in readlines(f)
            if line[1:10] == "capacities"
                items = split(line, "[")[2][1:end-2]
                capacities = parse.(Int, split(items, ", ")) 
            end
            if line[1:7] == "profits"
                items = split(line, "[")[2][1:end-2]
                profits = parse.(Int, split(items, ", "))
            end
            if line[1:7] == "weights"
                items = split(line, "[")[2][1:end-2]
                weights = parse.(Int, split(items, ", "))
            end
        end
        close(f)
        capacities = (capacities[1], capacities[2], capacities[3])
        println("Problem $(i-1):")
        println("capacities: ", capacities)
        println("profits: ", profits)
        println("weights: ", weights)
        kp = MultiKnapsackProblem(capacities, weights, profits)
        push!(problems, kp)
    end
    return problems
end

generate_problems (generic function with 1 method)

In [16]:
function test_with_buf(p, buf_len)
    x0 = fill(0, length(p.weights)) 
    st = TabuState(p, x0; buffer_length=buf_len)
    sol = solve_tabu(p, st; iteration_limit=100000)
    knapsack1 = []
    knapsack2 = []
    knapsack3 = []
    not_assigned = []
    for (i, k) in enumerate(sol)
        if k == 1
            push!(knapsack1, i)
        elseif k == 2
            push!(knapsack2, i)
        elseif k == 3
            push!(knapsack3, i)
        else
            push!(not_assigned, i)
        end
    end
    println("Items in knapsack 1: ", knapsack1)
    println("Items in knapsack 2: ", knapsack2)
    println("Items in knapsack 3: ", knapsack3)
    println("Items not assigned: ", not_assigned)
    println("Best objective: ", st.best_seen_obj)
    println("Last iteration: ", st.iter)
end

problems_from_files = generate_problems()

Problem 0:
capacities: (619, 360, 984)
profits: [830, 823, 288, 56, 726, 439, 534, 194, 387, 749, 127, 846, 165, 868, 143, 419, 750, 283, 505, 435, 828, 946, 315, 559, 462, 228, 342, 400, 371, 576, 356, 807, 346, 591, 991, 311, 878, 104, 318, 347, 985, 345, 354, 939, 333, 46, 873, 640, 947, 341, 746, 806, 609, 833, 731, 922, 404, 949, 916, 138, 715, 373, 997, 57, 659, 887, 797, 854, 537, 342, 412, 671, 406, 275, 846, 570, 142, 587, 734, 266]
weights: [61, 75, 95, 19, 77, 20, 75, 12, 49, 90, 24, 48, 84, 100, 33, 19, 20, 41, 100, 38, 64, 91, 16, 35, 45, 47, 45, 83, 98, 41, 13, 71, 46, 68, 59, 62, 19, 79, 48, 59, 100, 17, 31, 45, 40, 75, 57, 83, 57, 79, 35, 69, 40, 94, 39, 65, 11, 56, 63, 21, 22, 22, 59, 79, 13, 10, 72, 59, 52, 59, 52, 27, 92, 55, 44, 42, 64, 32, 85, 68]
Problem 1:
capacities: (985, 534, 988)
profits: [675, 262, 976, 36, 613]
weights: [20, 34, 65, 11, 26]
Problem 2:
capacities: (919, 873, 624)
profits: [541, 542, 847, 338, 306, 729, 25, 368, 864, 461]
weights: [78, 84, 48

10-element Vector{Any}:
 MultiKnapsackProblem((619, 360, 984), [61, 75, 95, 19, 77, 20, 75, 12, 49, 90  …  52, 27, 92, 55, 44, 42, 64, 32, 85, 68], [830, 823, 288, 56, 726, 439, 534, 194, 387, 749  …  412, 671, 406, 275, 846, 570, 142, 587, 734, 266])
 MultiKnapsackProblem((985, 534, 988), [20, 34, 65, 11, 26], [675, 262, 976, 36, 613])
 MultiKnapsackProblem((919, 873, 624), [78, 84, 48, 71, 40, 90, 31, 45, 44, 32], [541, 542, 847, 338, 306, 729, 25, 368, 864, 461])
 MultiKnapsackProblem((661, 880, 553), [96, 94, 82, 87, 57, 25, 54, 37, 61, 98, 68, 48, 75, 93, 53, 97, 65, 20, 70, 90], [598, 680, 243, 391, 330, 531, 556, 452, 378, 885, 332, 758, 249, 539, 100, 344, 663, 442, 425, 595])
 MultiKnapsackProblem((256, 281, 766), [89, 10, 30, 35, 37, 65, 37, 31, 43, 68  …  67, 29, 15, 79, 75, 88, 41, 34, 22, 23], [983, 938, 671, 69, 703, 664, 700, 204, 356, 41  …  290, 475, 187, 36, 869, 124, 106, 949, 977, 113])
 MultiKnapsackProblem((671, 944, 851), [19, 16, 67, 82, 56, 80, 60, 55, 63, 92  

Tabusearch z trzema plecakami - długość listy zakazów: 1

In [ ]:

for (i, p) in enumerate(problems_from_files)
    println("------------KNAPSACK PROBLEM $(i-1)------------")
    test_with_buf(p, 1)
    println("--------------------------------------------")
end

------------KNAPSACK PROBLEM 0------------
Items in knapsack 1: Any[22, 35, 41, 44, 49, 56, 58, 59, 63, 65, 66]
Items in knapsack 2: Any[12, 14, 17, 37, 47, 57, 68, 75]
Items in knapsack 3: Any[1, 2, 5, 6, 10, 21, 32, 48, 51, 52, 54, 55, 61, 67, 72, 79]
Items not assigned: Any[3, 4, 7, 8, 9, 11, 13, 15, 16, 18, 19, 20, 23, 24, 25, 26, 27, 28, 29, 30, 31, 33, 34, 36, 38, 39, 40, 42, 43, 45, 46, 50, 53, 60, 62, 64, 69, 70, 71, 73, 74, 76, 77, 78, 80]
Best objective: -28332
Last iteration: 100000
--------------------------------------------
------------KNAPSACK PROBLEM 1------------
Items in knapsack 1: Any[1, 2, 3, 4, 5]
Items in knapsack 2: Any[]
Items in knapsack 3: Any[]
Items not assigned: Any[]
Best objective: -2562
Last iteration: 100000
--------------------------------------------
------------KNAPSACK PROBLEM 2------------


Tabusearch z trzema plecakami - długość listy zakazów: 2

In [ ]:

for (i, p) in enumerate(problems_from_files)
    println("------------KNAPSACK PROBLEM $(i-1)------------")
    test_with_buf(p, 2)
    println("--------------------------------------------")
end

Tabusearch z trzema plecakami - długość listy zakazów: 5

In [ ]:
for (i, p) in enumerate(problems_from_files)
    println("------------KNAPSACK PROBLEM $(i-1)------------")
    test_with_buf(p, 5)
    println("--------------------------------------------")
end

In [ ]:
mutable struct TabuState{TMove,P,TF}
    tabu_buffer::CircularBuffer{TMove}
    best_seen::P
    best_seen_obj::TF
    current::P
    considered::P
    iter::Int
end

function TabuState(p, x0; buffer_length::Int=10)
    moves = possible_moves(p, x0)
    obj = objective(p, x0)
    return TabuState{eltype(moves),typeof(x0),typeof(obj)}(
        CircularBuffer{eltype(moves)}(buffer_length), x0, obj, copy(x0), copy(x0), 1
    )
end


function solve_tabu(p, s::TabuState; iteration_limit::Int=100)
    while s.iter < iteration_limit
        moves = possible_moves(p, s.current)
        best_move = 0
        best_move_obj = Inf
        for (i_move, move) in enumerate(moves)
            if in(move, s.tabu_buffer)
                # move forbidden, do not consider
                continue
            end
            # evaluate move
            copyto!(s.considered, s.current)
            apply!(s.considered, move)
            considered_value = objective(p, s.considered)
            if considered_value < best_move_obj
                best_move = i_move
                best_move_obj = considered_value
            end
        end
        # no allowed move found
        if best_move == 0
            break
        end
        apply!(s.current, moves[best_move])
        push!(s.tabu_buffer, invert_move(p, moves[best_move]))
        if best_move_obj < s.best_seen_obj
            # best so far, let's remember it
            copyto!(s.best_seen, s.current)
            s.best_seen_obj = best_move_obj
        end
        s.iter += 1
    end
    return s.best_seen
end

struct MultiKnapsackProblem
    capacities::NTuple{3,Int}
    weights::Vector{Int}
    profits::Vector{Int}
end

function objective(p::MultiKnapsackProblem, x::Vector{Int})
    total_weight = (0, 0, 0)
    total_profit = 0
    for (i, assign) in enumerate(x)
        if assign != 0
            total_weight = ntuple(j -> total_weight[j] + (j == assign ? p.weights[i] : 0), 3)
            total_profit += p.profits[i]
        end
    end
    for i in 1:3
        if total_weight[i] > p.capacities[i]
            return Inf  # Przekroczenie pojemności = niedozwolone
        end
    end
    return -total_profit  # minimalizujemy, więc negujemy
end

function apply!(x::Vector{Int}, move::Tuple{Symbol,Int,Int})
    action, item, knapsack = move
    if action == :assign
        x[item] = knapsack
    elseif action == :unassign
        x[item] = 0
    end
    return x
end

function invert_move(p::MultiKnapsackProblem, move::Tuple{Symbol,Int,Int})
    action, item, knapsack = move
    if action == :assign
        return (:unassign, item, knapsack)
    elseif action == :unassign
        return (:assign, item, knapsack)
    end
end

function possible_moves(p::MultiKnapsackProblem, x::Vector{Int})
    move_list = Tuple{Symbol,Int,Int}[]
    current_weights = (0, 0, 0)
    for (i, assign) in enumerate(x)
        if assign != 0
            current_weights = ntuple(j -> current_weights[j] + (j == assign ? p.weights[i] : 0), 3)
        end
    end

    for (i, assign) in enumerate(x)
        if assign == 0
            for k in 1:3
                if current_weights[k] + p.weights[i] <= p.capacities[k]
                    push!(move_list, (:assign, i, k))
                end
            end
        else
            push!(move_list, (:unassign, i, assign))
        end
    end
    return move_list
end

function generate_problem()
    n_items = 100
    profits = rand(DiscreteUniform(10, 1000), n_items)
    weights = rand(DiscreteUniform(10, 100), n_items)
    capacities = (1500, 1500, 1500)
    return MultiKnapsackProblem(capacities, weights, profits)
end

function test(p)
    x0 = fill(0, length(p.weights))  # 0 oznacza brak przypisania
    st = TabuState(p, x0; buffer_length=20)
    sol = solve_tabu(p, st; iteration_limit=100000)
    for (i, k) in enumerate(sol)
        if k != 0
            println("Item $i in knapsack $k")
        end
    end
    println("Best objective: ", st.best_seen_obj)
    println("Last iteration: ", st.iter)
end

# Run test
p = generate_problem()
test(p)
